# 🤖 Hey Nova — Wake Word Training Pipeline
### LagmaBills Robot — Training su Google Colab

**Ordine celle:**
1. Setup & GPU check
2. Installazione dipendenze
3. Configurazione
4. Genera dataset positivo (TTS ~20k campioni)
5. Scarica dataset negativi (MUSAN + hard negatives)
6. Augmentation
7. Feature extraction + Training
8. Plot training curves
9. Ottimizzazione TFLite INT8
10. Download modello → copia su RPi

> ⚡ **Prima di iniziare:** `Runtime > Cambia tipo di runtime > GPU (T4)`

In [ ]:
# CELLA 1 — Verifica GPU e monta Google Drive
import subprocess, sys

# Verifica GPU
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() if result.returncode == 0 else 'Non disponibile')

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU TF:', tf.config.list_physical_devices('GPU'))

# Monta Google Drive per salvare modello finale
from google.colab import drive
drive.mount('/content/drive')
print('Drive montato')

In [ ]:
# CELLA 2 — Installazione dipendenze
# TensorFlow e la maggior parte delle librerie sono già su Colab
!pip install -q edge-tts librosa soundfile openwakeword pyroomacoustics tqdm pyyaml
print('Dipendenze installate')

In [ ]:
# CELLA 3 — Configurazione
import yaml
from pathlib import Path

CFG = {
    'wake_word': {
        'name': 'nova',
        'phonemes': 'noUv@',
        'similar_words': ['nova', 'bova', 'rova', 'nuova', 'lova', 'cova']
    },
    'audio': {'sample_rate': 16000, 'chunk_size': 1280, 'channels': 1,
              'dtype': 'int16', 'bit_depth': 16},
    'augmentation': {
        'noise': {'snr_range_db': [5, 30]},
        'reverb': {'room_scale': [0.1, 0.5], 'wet_level': [0.0, 0.3]},
    },
    'training': {
        'batch_size': 256, 'epochs': 100, 'learning_rate': 0.001,
        'early_stopping_patience': 10, 'val_split': 0.1, 'positive_weight': 5.0
    },
    'inference': {
        'threshold': 0.70, 'smoothing_window': 4,
        'refractory_period_s': 2.0, 'vad_enabled': True,
        'vad_threshold': 0.5, 'debug_mode': False
    }
}

for d in ['data/positive','data/negative/musan','data/negative/hard_negatives',
          'data/augmented','models']:
    Path(d).mkdir(parents=True, exist_ok=True)

with open('config.yaml', 'w') as f:
    yaml.dump(CFG, f, default_flow_style=False, allow_unicode=True)

print(f"Config OK — wake word: '{CFG['wake_word']['name']}'")

In [ ]:
# CELLA 4 — Genera dataset POSITIVO (~20.000 campioni 'Hey Nova')
# Tempo stimato: 15-30 minuti
import os, asyncio, random, tempfile
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from tqdm.notebook import tqdm

SR = 16000
OUT_DIR = Path('data/positive')

# "hey nova" ha peso doppio: è la wake word esatta → ~50% dei campioni
# Le altre frasi coprono variazioni naturali del parlato
PHRASES_WEIGHTED = (
    ['hey {w}'] * 10 +          # 50% — frase target principale
    ['hey {w}'] * 0 +           # placeholder per leggibilità
    ['ehi {w}'] * 3 +           # 15%
    ['ok {w}'] * 2 +            # 10%
    ['ciao {w}'] * 2 +          # 10%
    ['{w} ascoltami'] * 1 +     #  5%
    ['{w} svegliati'] * 1 +     #  5%
    ['su {w}'] * 1              #  5%
)

VOICES = [
    'it-IT-ElsaNeural','it-IT-IsabellaNeural','it-IT-DiegoNeural',
    'en-US-JennyNeural','en-US-GuyNeural','en-GB-SoniaNeural',
    'en-GB-RyanNeural','de-DE-KatjaNeural','fr-FR-DeniseNeural','es-ES-ElviraNeural'
]

async def gen_edge(text, voice, out_path, rate='+0%', pitch='+0Hz'):
    try:
        import edge_tts
        comm = edge_tts.Communicate(text=text, voice=voice, rate=rate, pitch=pitch)
        with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as tmp:
            tmp_path = tmp.name
        await comm.save(tmp_path)
        audio, _ = librosa.load(tmp_path, sr=SR, mono=True)
        sf.write(out_path, audio, SR, subtype='PCM_16')
        os.unlink(tmp_path)
        return True
    except Exception:
        return False

def normalize(audio, target_db=-23.0):
    rms = np.sqrt(np.mean(audio**2))
    return audio * (10**(target_db/20)/rms) if rms > 1e-8 else audio

def pad_silence(audio, sr, pre=200, post=200):
    return np.concatenate([np.zeros(int(sr*pre/1000)), audio, np.zeros(int(sr*post/1000))])

async def generate_dataset(target_n=20000):
    generated, errors, idx = 0, 0, 0
    pbar = tqdm(total=target_n, desc='Generazione positivi')
    while generated < target_n:
        phrase = random.choice(PHRASES_WEIGHTED).format(w='nova')
        sp = random.randint(-15, 15)
        ph = random.randint(-20, 20)
        ok = await gen_edge(
            phrase, random.choice(VOICES),
            str(OUT_DIR / f'pos_{idx:06d}.wav'),
            f"{'+' if sp>=0 else ''}{sp}%", f"{'+' if ph>=0 else ''}{ph}Hz"
        )
        if ok:
            if random.random() < 0.5:
                try:
                    a, sr2 = sf.read(str(OUT_DIR / f'pos_{idx:06d}.wav'))
                    if len(a.shape)>1: a=a.mean(axis=1)
                    if random.random()<0.4:
                        a = librosa.effects.pitch_shift(a, sr=sr2, n_steps=random.uniform(-3,3))
                    a = normalize(a, random.uniform(-28,-18))
                    a = pad_silence(a, sr2, random.randint(50,300), random.randint(50,300))
                    sf.write(str(OUT_DIR / f'pos_{idx:06d}.wav'), a, SR, subtype='PCM_16')
                except: pass
            generated += 1
            pbar.update(1)
        else:
            errors += 1
            if errors > 200: print('Troppi errori'); break
        idx += 1
    pbar.close()
    # Verifica distribuzione frasi (debug)
    hey_count = sum(1 for p in Path('data/positive').glob('*.wav') if True)
    print(f'Generati {generated} campioni positivi (errori: {errors})')
    print(f'Distribuzione target: ~50% "hey nova", ~50% variazioni')

await generate_dataset(20000)


In [ ]:
# CELLA 5 — Scarica dataset NEGATIVI
# Usa solo il subset NOISE di MUSAN (~1 GB invece di 11 GB)
# Evita timeout su sessioni Colab free e velocizza il download
from pathlib import Path
import random

MUSAN_NEG_DIR = Path('data/negative/musan')
MUSAN_NEG_DIR.mkdir(parents=True, exist_ok=True)

print('Download MUSAN — solo subset noise (~1 GB)...')
!wget -q --show-progress -O /tmp/musan_noise.tar.gz \
    https://www.openslr.org/resources/17/musan_noise.tar.gz
print('Estrazione MUSAN noise...')
!tar -xzf /tmp/musan_noise.tar.gz -C data/negative/
!rm /tmp/musan_noise.tar.gz

# Copia i file nella cartella attesa dal resto del notebook
import shutil
noise_src = Path('data/negative/musan/noise')
if noise_src.exists():
    for fp in noise_src.rglob('*.wav'):
        shutil.copy(fp, MUSAN_NEG_DIR / fp.name)
    print(f'MUSAN noise: {len(list(MUSAN_NEG_DIR.glob("*.wav")))} file')
else:
    print(f'MUSAN: {len(list(Path("data/negative/musan").rglob("*.wav")))} file')

# Hard negatives: parole simili a 'nova' — critiche per ridurre falsi positivi
HARD_NEG = ['nuova','cova','bova','rova','trova','prova',
             'giova','piova','nove','note','notte','nome','nora','noia','dove']

hard_dir = Path('data/negative/hard_negatives')
hard_dir.mkdir(parents=True, exist_ok=True)

async def gen_hard_negatives():
    from tqdm.notebook import tqdm
    idx = 0
    pbar = tqdm(total=len(HARD_NEG)*50, desc='Hard negatives')
    for word in HARD_NEG:
        for _ in range(50):
            sp = random.randint(-10, 10)
            ok = await gen_edge(
                word, random.choice(VOICES),
                str(hard_dir / f'hardneg_{idx:05d}.wav'),
                rate=f"{'+' if sp>=0 else ''}{sp}%"
            )
            if ok: idx+=1; pbar.update(1)
    pbar.close()
    print(f'Hard negatives: {idx} file')

await gen_hard_negatives()


In [ ]:
# CELLA 6 — Augmentation dataset positivo (x3)
# Tempo stimato: 20-40 minuti
import numpy as np, soundfile as sf, random, warnings
from pathlib import Path
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

SR = 16000
AUG_DIR = Path('data/augmented')

# Carica noise bank da MUSAN
print('Caricamento noise bank...')
_noise = []
for fp in tqdm(list(Path('data/negative/musan').rglob('*.wav'))[:300], desc='Noise bank'):
    try:
        a, sr2 = sf.read(str(fp))
        if len(a.shape)>1: a=a.mean(axis=1)
        if sr2 != SR: a = librosa.resample(a, orig_sr=sr2, target_sr=SR)
        if len(a) > SR*2: _noise.append(a.astype(np.float32))
    except: pass
if not _noise: _noise = [np.random.randn(SR*5).astype(np.float32)]
print(f'Noise bank: {len(_noise)} clip')

def add_noise(audio, snr_db):
    n = random.choice(_noise)
    if len(n)<len(audio): n=np.tile(n,(len(audio)//len(n))+1)
    s=random.randint(0,max(0,len(n)-len(audio)-1))
    n=n[s:s+len(audio)]
    sp=np.mean(audio**2); np2=np.mean(n**2)
    if np2<1e-10 or sp<1e-10: return audio
    return np.clip(audio+np.sqrt(sp/(np2*10**(snr_db/10)))*n,-1.,1.)

def reverb(audio, wet=0.15):
    out=audio.copy()
    for ms,dec in [(30,.4),(60,.2),(120,.1)]:
        d=int(SR*ms/1000); delay=np.zeros_like(audio)
        delay[d:]=audio[:-d]*dec; out=out+wet*delay
    return np.clip(out,-1.,1.)

def augment(audio):
    audio=audio.astype(np.float32)
    mx=np.max(np.abs(audio))
    if mx>0: audio=audio/mx*.9
    if random.random()<0.6: audio=reverb(audio, random.uniform(.05,.25))
    if random.random()<0.4: audio=audio*(10**(random.uniform(-15,0)/20))
    if random.random()<0.8: audio=add_noise(audio, random.uniform(5,30))
    return audio

MULTIPLIER = 3
pos_files = sorted(Path('data/positive').glob('*.wav'))
print(f'Augmentation: {len(pos_files)} x {MULTIPLIER} = {len(pos_files)*MULTIPLIER} nuovi campioni')
idx=0
for fp in tqdm(pos_files, desc='Augmenting'):
    try:
        a, _ = sf.read(str(fp))
        if len(a.shape)>1: a=a.mean(axis=1)
        for _ in range(MULTIPLIER):
            sf.write(str(AUG_DIR/f'aug_{idx:07d}.wav'), augment(a.copy()), SR, subtype='PCM_16')
            idx+=1
    except: pass
print(f'Generati {idx} campioni augmentati')

In [ ]:
# CELLA 7 — Feature extraction + Training
# Tempo stimato: 20-60 minuti (GPU)
import numpy as np, soundfile as sf, tensorflow as tf, openwakeword, random
from pathlib import Path
from tqdm.notebook import tqdm

SR, CHUNK = 16000, 1280
MODELS_DIR = Path('models')

print(f'TF: {tf.__version__} | GPU: {tf.config.list_physical_devices("GPU")}')

print('Caricamento backbone openWakeWord...')
oww = openwakeword.Model(wakeword_models=[], enable_speex_noise_suppression=False)

def extract_features(files, label, desc):
    X, y = [], []
    for fp in tqdm(files, desc=desc, leave=False):
        try:
            a, sr = sf.read(str(fp))
            if len(a.shape)>1: a=a.mean(axis=1)
            a16=(a*32767).astype(np.int16)
            for i in range(0, len(a16)-CHUNK, CHUNK//2):
                chunk=a16[i:i+CHUNK]
                if len(chunk)==CHUNK:
                    feat=oww.get_parent_model_output(chunk)
                    if feat is not None and len(feat)>0:
                        X.append(feat.flatten()); y.append(label)
        except: pass
    return X, y

pos_files = sorted(Path('data/positive').glob('*.wav')) + sorted(Path('data/augmented').glob('*.wav'))
neg_files = list(Path('data/negative/musan').rglob('*.wav')) + list(Path('data/negative/hard_negatives').rglob('*.wav'))
random.shuffle(neg_files)
neg_files = neg_files[:len(pos_files)*5]
print(f'File: {len(pos_files)} pos, {len(neg_files)} neg')

Xp, yp = extract_features(pos_files, 1, 'Features POS')
Xn, yn = extract_features(neg_files, 0, 'Features NEG')

X = np.array(Xp+Xn, dtype=np.float32)
y = np.array(yp+yn, dtype=np.float32)
idx2 = np.random.permutation(len(X))
X, y = X[idx2], y[idx2]
val = int(len(X)*.1)
X_train, X_val, y_train, y_val = X[val:], X[:val], y[val:], y[:val]
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Pos: {y_train.sum():.0f} ({y_train.mean()*100:.1f}%)')

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X.shape[1],)),
    tf.keras.layers.Dense(128), tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'), tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64), tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation('relu'), tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(32), tf.keras.layers.Activation('relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
], name='heynova_dnn')
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_auc', patience=10, mode='max',
                                      restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(str(MODELS_DIR/'best_model.h5'),
                                        monitor='val_auc', mode='max', save_best_only=True),
]

history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    batch_size=256, epochs=100, class_weight={0:1.0,1:5.0},
    callbacks=callbacks, verbose=1
)

results = model.evaluate(X_val, y_val, verbose=0)
metrics = dict(zip(model.metrics_names, results))
y_pred = model.predict(X_val, verbose=0).flatten()
fpr = ((y_pred>0.5)&(y_val==0)).sum()/(y_val==0).sum()
print(f'AUC:{metrics["auc"]:.4f}  Precision:{metrics["precision"]:.4f}  Recall:{metrics["recall"]:.4f}  FPR:{fpr*100:.2f}%')
model.save(str(MODELS_DIR/'heynova_model.h5'))
print('Modello salvato: models/heynova_model.h5')

In [ ]:
# CELLA 8 — Plot training curves
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Hey Nova — Training Curves', fontsize=14)
axes[0,0].plot(history.history['loss'],label='Train'); axes[0,0].plot(history.history['val_loss'],label='Val'); axes[0,0].set_title('Loss'); axes[0,0].legend()
axes[0,1].plot(history.history['auc'],label='Train'); axes[0,1].plot(history.history['val_auc'],label='Val'); axes[0,1].set_title('AUC'); axes[0,1].legend()
axes[1,0].plot(history.history['precision'],label='Precision'); axes[1,0].plot(history.history['recall'],label='Recall'); axes[1,0].set_title('Precision/Recall'); axes[1,0].legend()
axes[1,1].plot(history.history['accuracy'],label='Train'); axes[1,1].plot(history.history['val_accuracy'],label='Val'); axes[1,1].set_title('Accuracy'); axes[1,1].legend()
plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=150)
plt.show()
print('Plot salvato')

In [ ]:
# CELLA 9 — Ottimizzazione TFLite INT8 per RPi
import tensorflow as tf, numpy as np, soundfile as sf, openwakeword
from pathlib import Path

MODELS_DIR = Path('models')
model = tf.keras.models.load_model(str(MODELS_DIR/'heynova_model.h5'))

# Ricarica oww qui — necessario se la cella viene eseguita senza la 7
# (oww non è in scope se riavvii solo questa cella)
print('Caricamento backbone openWakeWord per calibrazione...')
oww = openwakeword.Model(wakeword_models=[], enable_speex_noise_suppression=False)

# Dataset calibrazione per quantizzazione INT8
print('Preparazione calibrazione...')
calib = []
for fp in list(Path('data/positive').glob('*.wav'))[:300]:
    try:
        a, _ = sf.read(str(fp))
        if len(a.shape)>1: a=a.mean(axis=1)
        a16=(a*32767).astype(np.int16)
        if len(a16)>=1280:
            feat=oww.get_parent_model_output(a16[:1280])
            if feat is not None: calib.append(feat.flatten().astype(np.float32))
    except: pass
calib_data = np.array(calib)
print(f'Campioni calibrazione: {len(calib_data)}')

# ── Versione INT8 FULL (input/output rimangono float32 lato Python) ──────────
# NOTA: inference_input_type e inference_output_type vengono OMESSI di proposito.
# Con tf.int8 il runtime si aspetterebbe input già quantizzato — difficile da
# gestire in PyAudio. Usiamo full-integer quantization solo sui pesi interni:
# il modello rimane comodo da chiamare con float32 e pesa comunque ~4x meno.
conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
def rep_data():
    for s in calib_data[:200]: yield [s.reshape(1,-1)]
conv.representative_dataset = rep_data
print('Conversione INT8 (pesi quantizzati, I/O float32)...')
m = conv.convert()
with open(str(MODELS_DIR/'heynova_int8.tflite'),'wb') as f: f.write(m)
print(f'heynova_int8.tflite: {len(m)/1024:.1f} KB')

# ── Versione Dynamic quantization (fallback più semplice) ────────────────────
conv2 = tf.lite.TFLiteConverter.from_keras_model(model)
conv2.optimizations = [tf.lite.Optimize.DEFAULT]
m2 = conv2.convert()
with open(str(MODELS_DIR/'heynova_dynamic.tflite'),'wb') as f: f.write(m2)
print(f'heynova_dynamic.tflite: {len(m2)/1024:.1f} KB')
print('Modelli pronti per RPi! Usa heynova_int8.tflite per inferenza.')


In [ ]:
# CELLA 10 — Salva su Google Drive + Download diretto
import shutil
from google.colab import files
from pathlib import Path

# Salva su Drive
drive_out = Path('/content/drive/MyDrive/LagmaBills/wakeword_models')
drive_out.mkdir(parents=True, exist_ok=True)
for fname in ['heynova_int8.tflite','heynova_dynamic.tflite','heynova_model.h5','training_curves.png']:
    src = Path('models') / fname
    if src.exists():
        shutil.copy(src, drive_out/fname)
        print(f'Drive: {fname}')
shutil.copy('config.yaml', drive_out/'config.yaml')
print(f'Salvato in: Drive/LagmaBills/wakeword_models/')

# Download diretto browser
files.download('models/heynova_int8.tflite')
files.download('config.yaml')
print('Download avviato!')

## Dopo il download — Copia sul RPi

Dal terminale del PC:
```bash
ssh ladrodirame@LagmaBills.local "mkdir -p ~/wakeword/models"
scp heynova_int8.tflite ladrodirame@LagmaBills.local:~/wakeword/models/
scp config.yaml ladrodirame@LagmaBills.local:~/wakeword/
scp 5_realtime_inference.py ladrodirame@LagmaBills.local:~/wakeword/
scp 6_vad_pipeline.py ladrodirame@LagmaBills.local:~/wakeword/
```

Sul RPi:
```bash
pip install openwakeword tflite-runtime webrtcvad pyaudio --break-system-packages
# Test con debug visivo dello score
python 5_realtime_inference.py --model models/heynova_int8.tflite --debug
```

> Dì **"Hey Nova"** al microfono ReSpeaker — dovresti vedere la detection!